# 4. Comparação de modelos — Cenário B

Este notebook avalia se a inclusão de informações desagregadas da taxa de
evasão no período anterior acrescenta capacidade preditiva em relação ao
conjunto de informações agregadas utilizado no Cenário A.

Os modelos, protocolos temporais e métricas são mantidos iguais aos utilizados
no Cenário A.

## 4.1 Configuração

In [1]:
from pathlib import Path
import sys
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "src").is_dir()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Raiz do projeto: {PROJECT_ROOT}")

Raiz do projeto: /home/sara/Documentos/tcc-evasao-ensino-superior


In [3]:
from src.validation import criar_folds_temporais

from src.evaluation import (
    avaliar_persistencia,
    avaliar_modelo,
    criar_linear_regression,
    criar_ridge,
    criar_random_forest,
    criar_gradient_boosting
)

## 4.2 Carregamento da base do Cenário B

In [4]:
CAMINHO_CENARIO_B = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "base_modelo_uf_cenario_b.csv"
)

df_cenario_b = pd.read_csv(
    CAMINHO_CENARIO_B
)

print(
    "Dimensões:",
    df_cenario_b.shape
)

Dimensões: (189, 23)


In [5]:
df_cenario_b["ano_inicio"] = (
    df_cenario_b["ano_fluxo"]
    .str[:4]
    .astype(int)
)

## 4.3 Definição das variáveis preditoras

In [6]:
features_cenario_b = [
    # Histórico agregado
    "evasao_t_1",
    "conclusao_t_1",
    "retencao_t_1",
    "permanencia_t_1",

    # Sexo
    "evasao_feminino_t_1",
    "evasao_masculino_t_1",

    # PPI
    "evasao_ppi_sim_t_1",
    "evasao_ppi_nao_t_1",

    # Faixa etária
    "evasao_ate_19_t_1",
    "evasao_20_22_t_1",
    "evasao_23_24_t_1",
    "evasao_25_29_t_1",
    "evasao_30_39_t_1",
    "evasao_40_49_t_1",
    "evasao_50_mais_t_1",

    # Deficiência
    "evasao_deficiencia_sim_t_1",
    "evasao_deficiencia_nao_t_1"
]

print(
    "Número de features:",
    len(features_cenario_b)
)

Número de features: 17


In [7]:
features_ausentes = [
    coluna
    for coluna in features_cenario_b
    if coluna not in df_cenario_b.columns
]

print("Features ausentes:", features_ausentes)

Features ausentes: []


## 4.4 Protocolos de validação

In [8]:
folds_expanding = criar_folds_temporais(
    df_cenario_b,
    estrategia="expanding",
    tamanho_treino=4
)

folds_rolling = criar_folds_temporais(
    df_cenario_b,
    estrategia="rolling",
    tamanho_treino=4
)

In [9]:
print("Expanding:")
display(pd.DataFrame(folds_expanding))

print("\nRolling:")
display(pd.DataFrame(folds_rolling))

Expanding:


,fold,periodos_treino,periodo_teste,n_treino,n_teste,ufs_treino,ufs_teste
0,1,"[2017, 2018, 2019, 2020]",2021,108,27,27,27
1,2,"[2017, 2018, 2019, 2020, 2021]",2022,135,27,27,27
2,3,"[2017, 2018, 2019, 2020, 2021, 2022]",2023,162,27,27,27



Rolling:


,fold,periodos_treino,periodo_teste,n_treino,n_teste,ufs_treino,ufs_teste
0,1,"[2017, 2018, 2019, 2020]",2021,108,27,27,27
1,2,"[2018, 2019, 2020, 2021]",2022,108,27,27,27
2,3,"[2019, 2020, 2021, 2022]",2023,108,27,27,27


## 4.5 Baseline — Persistência

In [10]:
resultados_persistencia_expanding, previsoes_persistencia_expanding, metricas_persistencia_expanding = (
    avaliar_persistencia(
        df_cenario_b,
        folds_expanding
    )
)

metricas_persistencia_expanding

{'mae': 1.9839506172839505,
 'rmse': 2.5239837216512497,
 'r2': -0.23992513603135524}

In [11]:
resultados_persistencia_rolling, previsoes_persistencia_rolling, metricas_persistencia_rolling = (
    avaliar_persistencia(
        df_cenario_b,
        folds_rolling
    )
)

metricas_persistencia_rolling

{'mae': 1.9839506172839505,
 'rmse': 2.5239837216512497,
 'r2': -0.23992513603135524}

## 4.6 Regressão Linear

In [12]:
resultados_lr_expanding, previsoes_lr_expanding, metricas_lr_expanding = (
    avaliar_modelo(
        df_cenario_b,
        folds_expanding,
        criar_linear_regression,
        features=features_cenario_b
    )
)

metricas_lr_expanding

{'mae': 1.6712942235687667,
 'rmse': 2.156415265613147,
 'r2': 0.09491959984250098}

In [13]:
resultados_lr_rolling, previsoes_lr_rolling, metricas_lr_rolling = (
    avaliar_modelo(
        df_cenario_b,
        folds_rolling,
        criar_linear_regression,
        features=features_cenario_b
    )
)

metricas_lr_rolling

{'mae': 1.7276062550157043,
 'rmse': 2.142579935513556,
 'r2': 0.10649614181767131}

## 4.7 Ridge

In [14]:
resultados_ridge_expanding, previsoes_ridge_expanding, metricas_ridge_expanding = (
    avaliar_modelo(
        df_cenario_b,
        folds_expanding,
        criar_ridge,
        features=features_cenario_b
    )
)

metricas_ridge_expanding

{'mae': 1.6552949835644453,
 'rmse': 2.1557144612044463,
 'r2': 0.09550778086501599}

In [15]:
resultados_ridge_rolling, previsoes_ridge_rolling, metricas_ridge_rolling = (
    avaliar_modelo(
        df_cenario_b,
        folds_rolling,
        criar_ridge,
        features=features_cenario_b
    )
)

metricas_ridge_rolling

{'mae': 1.6732471132878717,
 'rmse': 2.1041619189109935,
 'r2': 0.13825121860986467}

## 4.8 Random Forest

In [16]:
resultados_rf_expanding, previsoes_rf_expanding, metricas_rf_expanding = (
    avaliar_modelo(
        df_cenario_b,
        folds_expanding,
        criar_random_forest,
        features=features_cenario_b
    )
)

metricas_rf_expanding

{'mae': 1.850445010230969,
 'rmse': 2.3207688747506166,
 'r2': -0.048301355758223696}

In [17]:
resultados_rf_rolling, previsoes_rf_rolling, metricas_rf_rolling = (
    avaliar_modelo(
        df_cenario_b,
        folds_rolling,
        criar_random_forest,
        features=features_cenario_b
    )
)

metricas_rf_rolling

{'mae': 1.862789373041223,
 'rmse': 2.317378359374253,
 'r2': -0.04524057223990141}

## 4.9 Gradient Boosting

In [18]:
resultados_gb_expanding, previsoes_gb_expanding, metricas_gb_expanding = (
    avaliar_modelo(
        df_cenario_b,
        folds_expanding,
        criar_gradient_boosting,
        features=features_cenario_b
    )
)

metricas_gb_expanding

{'mae': 1.7685931915606896,
 'rmse': 2.3054337729575987,
 'r2': -0.034493263473316604}

In [19]:
resultados_gb_rolling, previsoes_gb_rolling, metricas_gb_rolling = (
    avaliar_modelo(
        df_cenario_b,
        folds_rolling,
        criar_gradient_boosting,
        features=features_cenario_b
    )
)

metricas_gb_rolling

{'mae': 1.7866534459085495,
 'rmse': 2.3034553147493337,
 'r2': -0.03271847948108597}

## 4.10 Comparação dos resultados

In [20]:
resultados_cenario_b = pd.DataFrame([
    {
        "modelo": "Persistência",
        "protocolo": "Expanding",
        **metricas_persistencia_expanding
    },
    {
        "modelo": "Persistência",
        "protocolo": "Rolling",
        **metricas_persistencia_rolling
    },
    {
        "modelo": "Regressão Linear",
        "protocolo": "Expanding",
        **metricas_lr_expanding
    },
    {
        "modelo": "Regressão Linear",
        "protocolo": "Rolling",
        **metricas_lr_rolling
    },
    {
        "modelo": "Ridge",
        "protocolo": "Expanding",
        **metricas_ridge_expanding
    },
    {
        "modelo": "Ridge",
        "protocolo": "Rolling",
        **metricas_ridge_rolling
    },
    {
        "modelo": "Random Forest",
        "protocolo": "Expanding",
        **metricas_rf_expanding
    },
    {
        "modelo": "Random Forest",
        "protocolo": "Rolling",
        **metricas_rf_rolling
    },
    {
        "modelo": "Gradient Boosting",
        "protocolo": "Expanding",
        **metricas_gb_expanding
    },
    {
        "modelo": "Gradient Boosting",
        "protocolo": "Rolling",
        **metricas_gb_rolling
    }
])

resultados_cenario_b = resultados_cenario_b.rename(
    columns={
        "mae": "MAE",
        "rmse": "RMSE",
        "r2": "R2"
    }
)

resultados_cenario_b = resultados_cenario_b[
    [
        "modelo",
        "protocolo",
        "MAE",
        "RMSE",
        "R2"
    ]
].round(4)

resultados_cenario_b

,modelo,protocolo,MAE,RMSE,R2
0,Persistência,Expanding,1.9840,2.5240,-0.2399
1,Persistência,Rolling,1.9840,2.5240,-0.2399
2,Regressão Linear,Expanding,1.6713,2.1564,0.0949
3,Regressão Linear,Rolling,1.7276,2.1426,0.1065
4,Ridge,Expanding,1.6553,2.1557,0.0955
5,Ridge,Rolling,1.6732,2.1042,0.1383
6,Random Forest,Expanding,1.8504,2.3208,-0.0483
7,Random Forest,Rolling,1.8628,2.3174,-0.0452
8,Gradient Boosting,Expanding,1.7686,2.3054,-0.0345
9,Gradient Boosting,Rolling,1.7867,2.3035,-0.0327


In [21]:
assert abs(
    metricas_persistencia_expanding["mae"]
    - 1.983951
) < 1e-4

assert abs(
    metricas_persistencia_rolling["mae"]
    - 1.983951
) < 1e-4

print(
    "Baseline de persistência consistente com o Cenário A."
)

Baseline de persistência consistente com o Cenário A.


## 4.11 Exportação dos resultados do Cenário B

In [22]:
CAMINHO_PREVISOES_GB_B = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "previsoes_gb_cenario_b_expanding.csv"
)

previsoes_gb_expanding.to_csv(
    CAMINHO_PREVISOES_GB_B,
    index=False
)

print(
    f"Resultados do Cenário B salvos em:\n{CAMINHO_PREVISOES_GB_B}"
)

print("B:", previsoes_gb_expanding.shape)

Resultados do Cenário B salvos em:
/home/sara/Documentos/tcc-evasao-ensino-superior/results/tables/previsoes_gb_cenario_b_expanding.csv
B: (81, 7)


In [23]:
resultados_por_fold_cenario_b = pd.concat(
    [
        resultados_persistencia_expanding.assign(
            modelo="Persistência",
            protocolo="Expanding"
        ),
        resultados_persistencia_rolling.assign(
            modelo="Persistência",
            protocolo="Rolling"
        ),
        resultados_lr_expanding.assign(
            modelo="Regressão Linear",
            protocolo="Expanding"
        ),
        resultados_lr_rolling.assign(
            modelo="Regressão Linear",
            protocolo="Rolling"
        ),
        resultados_ridge_expanding.assign(
            modelo="Ridge",
            protocolo="Expanding"
        ),
        resultados_ridge_rolling.assign(
            modelo="Ridge",
            protocolo="Rolling"
        ),
        resultados_rf_expanding.assign(
            modelo="Random Forest",
            protocolo="Expanding"
        ),
        resultados_rf_rolling.assign(
            modelo="Random Forest",
            protocolo="Rolling"
        ),
        resultados_gb_expanding.assign(
            modelo="Gradient Boosting",
            protocolo="Expanding"
        ),
        resultados_gb_rolling.assign(
            modelo="Gradient Boosting",
            protocolo="Rolling"
        )
    ],
    ignore_index=True
)

resultados_por_fold_cenario_b = resultados_por_fold_cenario_b[
    [
        "modelo",
        "protocolo",
        "fold",
        "periodo_teste",
        "mae",
        "rmse",
        "r2",
        "n_treino",
        "n_teste"
    ]
].sort_values(
    [
        "periodo_teste",
        "protocolo",
        "modelo"
    ]
).reset_index(drop=True)

In [24]:
CAMINHO_FOLDS_B = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "resultados_por_fold_cenario_b.csv"
)

resultados_por_fold_cenario_b.to_csv(
    CAMINHO_FOLDS_B,
    index=False
)

print(
    f"Resultados por fold do Cenário B salvos em:\n"
    f"{CAMINHO_FOLDS_B}"
)

Resultados por fold do Cenário B salvos em:
/home/sara/Documentos/tcc-evasao-ensino-superior/results/tables/resultados_por_fold_cenario_b.csv
